In [ ]:
# === auto-inserted: bev-solving src on path ===
import sys, pathlib
_root = pathlib.Path.cwd()
while _root != _root.parent and not (_root / 'src' / 'geometry.py').exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))


# Colab Inference for `v6_dinov2_lss_bev_cleaned`

Notebook для Google Colab:
- скачивает/читает kaggle-safe dataset `letiti6e/bev-yandex`
- загружает `best.pt`, `ema_best.pt`, `config.json`, `merged_cleaned.csv`, `test_matched_val200_split.npz`
- подбирает global threshold на `val`
- инферит `test`
- собирает несколько `zip`-посылок


In [ ]:
# Если нужно, можно раскомментировать
# !pip install -q kaggle


In [ ]:
import os
import gc
import json
import math
import random
import hashlib
import zipfile
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image, ImageFile
from tqdm import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'


## Paths

Ожидается, что kaggle dataset уже добавлен в Colab/Notebook environment,
а артефакты модели загружены вручную в `MODEL_DIR`.


In [ ]:
# Подправь под свою сессию Colab
DATA_ROOT = Path('/content/bev-yandex')
MODEL_DIR = Path('/content/v6_artifacts')
OUT_DIR = Path('/content/v6_infer_outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_TRAIN = DATA_ROOT / 'autonomy_yandex_dataset_train'
DATA_VAL = DATA_ROOT / 'autonomy_yandex_dataset_val'
DATA_TEST = DATA_ROOT / 'autonomy_yandex_dataset_test'

BEST_PT = MODEL_DIR / 'best.pt'
EMA_BEST_PT = MODEL_DIR / 'ema_best.pt'
CONFIG_JSON = MODEL_DIR / 'config.json'
MERGED_CLEANED_CSV = MODEL_DIR / 'merged_cleaned.csv'
SPLIT_NPZ = MODEL_DIR / 'test_matched_val200_split.npz'

assert DATA_TRAIN.exists(), DATA_TRAIN
assert DATA_VAL.exists(), DATA_VAL
assert DATA_TEST.exists(), DATA_TEST
assert BEST_PT.exists(), BEST_PT
assert EMA_BEST_PT.exists(), EMA_BEST_PT
assert CONFIG_JSON.exists(), CONFIG_JSON
assert MERGED_CLEANED_CSV.exists(), MERGED_CLEANED_CSV
assert SPLIT_NPZ.exists(), SPLIT_NPZ

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))


In [ ]:
cfg_json = json.loads(CONFIG_JSON.read_text())
cfg = {
    'img_hw': tuple(cfg_json.get('img_hw', [392, 700])),
    'rover_emb_dim': int(cfg_json.get('rover_emb_dim', 8)),
    'rover_cond_dim': int(cfg_json.get('rover_cond_dim', 8)),
    'backbone_name': cfg_json.get('backbone_name', 'dinov2_vitb14'),
    'hub_repo': cfg_json.get('hub_repo', 'facebookresearch/dinov2'),
    'backbone_out_dim': int(cfg_json.get('backbone_out_dim', 768)),
    'patch_size': int(cfg_json.get('patch_size', 14)),
    'tap_layers': tuple(cfg_json.get('tap_layers', [2, 5, 8, 11])),
    'neck_dim': int(cfg_json.get('neck_dim', 128)),
    'context_dim': int(cfg_json.get('context_dim', 80)),
    'depth_bins': int(cfg_json.get('depth_bins', 24)),
    'depth_min': float(cfg_json.get('depth_min', 1.0)),
    'depth_max': float(cfg_json.get('depth_max', 80.0)),
    'world_z_min': float(cfg_json.get('world_z_min', -2.0)),
    'world_z_max': float(cfg_json.get('world_z_max', 4.5)),
    'bev_base_channels': int(cfg_json.get('bev_base_channels', 96)),
    'bev_gn_groups': int(cfg_json.get('bev_gn_groups', 8)),
    'use_amp': bool(cfg_json.get('use_amp', True)),
}
cfg


In [ ]:
CAMERA_NAMES = [
    '/camera/inner/frontal/middle',
    '/camera/inner/frontal/far',
    '/side/left/forward',
    '/side/right/forward',
]
INTRINSICS_NAMES = [n + '/intrinsic_params' for n in CAMERA_NAMES]
CAR2CAM_NAMES = [n + '/car_to_cam' for n in CAMERA_NAMES]
GT_NAME = 'gt_occupancy_grid'

BEV_H, BEV_W = 188, 126
BEV_RES = 0.8
X_RANGE = (0.0, BEV_H * BEV_RES)
Y_RANGE = (-BEV_W * BEV_RES / 2, BEV_W * BEV_RES / 2)

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


In [ ]:
# Reusable code now lives in src/. See README.md.
from src.geometry import load_info_with_root, remap_kaggle_paths, resolve_info_path
from src.utils.training import cleanup_cuda


In [ ]:
clean_info = pd.read_csv(MERGED_CLEANED_CSV)
split = np.load(SPLIT_NPZ)
train_idx = split['train_idx'].tolist()
val_idx = split['val_idx'].tolist()

clean_info = remap_kaggle_paths(clean_info, DATA_TRAIN, DATA_VAL, DATA_TEST)
train_info = remap_kaggle_paths(clean_info.iloc[train_idx].reset_index(drop=True).copy(), DATA_TRAIN, DATA_VAL, DATA_TEST)
val_info = remap_kaggle_paths(clean_info.iloc[val_idx].reset_index(drop=True).copy(), DATA_TRAIN, DATA_VAL, DATA_TEST)
test_info = remap_kaggle_paths(load_info_with_root(DATA_TEST, 'test'), DATA_TRAIN, DATA_VAL, DATA_TEST)

print('train:', len(train_info), 'val:', len(val_info), 'test:', len(test_info))
print(train_info['__data_root'].value_counts())


In [ ]:
from src.splits import build_rover_vocab_from_train, encode_rover

rover_vocab = build_rover_vocab_from_train(train_info)
print('num rover classes:', len(rover_vocab))


In [ ]:
# Reusable code now lives in src/. See README.md.
from src.data import BEVDatasetV4Clean


In [ ]:
from src.models.lss import ASPP2d, ConvGNAct, LSSViewTransform2D, ResidualBlock2d, _DINOv2MultiScaleBackbone, gn_groups

class StrongBEVEncoderDecoder(nn.Module):
    def __init__(self, in_c, base_c=96, groups=8):
        super().__init__()
        self.stem = nn.Sequential(ConvGNAct(in_c, base_c, 3, 1, 1, groups, True), ResidualBlock2d(base_c, base_c, 1, groups))
        self.down1 = nn.Sequential(ResidualBlock2d(base_c, base_c * 2, 2, groups), ResidualBlock2d(base_c * 2, base_c * 2, 1, groups))
        self.down2 = nn.Sequential(ResidualBlock2d(base_c * 2, base_c * 4, 2, groups), ResidualBlock2d(base_c * 4, base_c * 4, 1, groups))
        self.aspp = ASPP2d(base_c * 4, base_c * 4, (1,3,6), groups)
        self.up1 = nn.Sequential(ConvGNAct(base_c * 4 + base_c * 2, base_c * 2, 3, 1, 1, groups, True), ResidualBlock2d(base_c * 2, base_c * 2, 1, groups))
        self.up0 = nn.Sequential(ConvGNAct(base_c * 2 + base_c, base_c, 3, 1, 1, groups, True), ResidualBlock2d(base_c, base_c, 1, groups))
        self.head = nn.Conv2d(base_c, 1, 1)
    def forward(self, x):
        s0 = self.stem(x)
        s1 = self.down1(s0)
        s2 = self.down2(s1)
        b = self.aspp(s2)
        u1 = self.up1(torch.cat([F.interpolate(b, size=s1.shape[-2:], mode='bilinear', align_corners=False), s1], dim=1))
        u0 = self.up0(torch.cat([F.interpolate(u1, size=s0.shape[-2:], mode='bilinear', align_corners=False), s0], dim=1))
        return self.head(u0)

class MultiCamBEVv6DINOv2LSSClean(nn.Module):
    def __init__(self, num_rover_classes, rover_emb_dim=8, rover_cond_dim=8, n_cameras=4, freeze_backbone=False, hub_repo='facebookresearch/dinov2', backbone_name='dinov2_vitb14', backbone_out_dim=768, patch_size=14, tap_layers=(2,5,8,11), neck_dim=128, context_dim=80, depth_bins=24, depth_min=1.0, depth_max=80.0, world_z_min=-2.0, world_z_max=4.5, bev_base_channels=96, bev_gn_groups=8):
        super().__init__()
        self.n_cameras = n_cameras
        self.rover_cond_dim = rover_cond_dim
        self.backbone = _DINOv2MultiScaleBackbone(hub_repo, backbone_name, backbone_out_dim, patch_size, tap_layers, neck_dim, bev_gn_groups)
        self.view_transform = LSSViewTransform2D(neck_dim, context_dim, depth_bins, depth_min, depth_max, BEV_H, BEV_W, BEV_RES, X_RANGE, Y_RANGE, world_z_min, world_z_max, bev_gn_groups)
        self.rover_embed = nn.Embedding(num_rover_classes, rover_emb_dim)
        self.rover_mlp = nn.Sequential(nn.Linear(rover_emb_dim, 16), nn.ReLU(inplace=True), nn.Linear(16, rover_cond_dim), nn.ReLU(inplace=True))
        self.bev_decoder = StrongBEVEncoderDecoder(context_dim + rover_cond_dim, bev_base_channels, bev_gn_groups)

    def forward(self, images, intrinsics, car2cams, rover_ids):
        B, N, C, H, W = images.shape
        x = images.reshape(B * N, C, H, W)
        back = self.backbone(x)
        fused = back['fused'].reshape(B, N, back['fused'].shape[1], back['fused'].shape[2], back['fused'].shape[3])
        bev = self.view_transform(fused, intrinsics, car2cams, image_hw=(H, W))
        rover_feat = self.rover_mlp(self.rover_embed(rover_ids)).view(B, self.rover_cond_dim, 1, 1)
        rover_map = rover_feat.expand(-1, -1, BEV_H, BEV_W)
        logits = self.bev_decoder(torch.cat([bev, rover_map], dim=1))
        return torch.nan_to_num(logits, nan=0.0, posinf=0.0, neginf=0.0)


In [ ]:
def build_model():
    return MultiCamBEVv6DINOv2LSSClean(
        num_rover_classes=len(rover_vocab),
        rover_emb_dim=cfg['rover_emb_dim'],
        rover_cond_dim=cfg['rover_cond_dim'],
        freeze_backbone=False,
        hub_repo=cfg['hub_repo'],
        backbone_name=cfg['backbone_name'],
        backbone_out_dim=cfg['backbone_out_dim'],
        patch_size=cfg['patch_size'],
        tap_layers=cfg['tap_layers'],
        neck_dim=cfg['neck_dim'],
        context_dim=cfg['context_dim'],
        depth_bins=cfg['depth_bins'],
        depth_min=cfg['depth_min'],
        depth_max=cfg['depth_max'],
        world_z_min=cfg['world_z_min'],
        world_z_max=cfg['world_z_max'],
        bev_base_channels=cfg['bev_base_channels'],
        bev_gn_groups=cfg['bev_gn_groups'],
    ).to(device)


def load_model_from_ckpt(ckpt_path: Path, use_ema_if_present: bool):
    ckpt = torch.load(ckpt_path, map_location='cpu')
    model = build_model()
    state = ckpt['ema'] if (use_ema_if_present and 'ema' in ckpt) else ckpt['model']
    missing, unexpected = model.load_state_dict(state, strict=False)
    print('loaded from:', ckpt_path.name)
    print('using state:', 'ema' if (use_ema_if_present and 'ema' in ckpt) else 'model')
    print('missing:', len(missing), 'unexpected:', len(unexpected))
    return model


In [ ]:
@torch.inference_mode()
def collect_val_probs(model, loader, cache_path: Path):
    if cache_path.exists():
        d = torch.load(cache_path, map_location='cpu')
        return d['probs'], d['gt']
    model.eval()
    probs_list, gt_list = [], []
    for batch in tqdm(loader, desc=f'collect val probs -> {cache_path.name}'):
        images = batch['images'].to(device)
        intr = batch['intrinsics'].to(device)
        c2c = batch['car2cams'].to(device)
        rover_id = batch['rover_id'].to(device)
        gt = batch['gt'].cpu()
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda' and cfg['use_amp'])):
            logits = model(images, intr, c2c, rover_id)
        probs = torch.sigmoid(logits.float()).cpu().to(torch.float16)
        probs_list.append(probs)
        gt_list.append(gt)
    probs = torch.cat(probs_list, dim=0)
    gt = torch.cat(gt_list, dim=0)
    torch.save({'probs': probs, 'gt': gt}, cache_path)
    return probs, gt


def threshold_sweep_from_cached_probs(probs, gt, thresholds):
    inter = torch.zeros(len(thresholds), dtype=torch.float64)
    union = torch.zeros(len(thresholds), dtype=torch.float64)
    valid = gt != 255
    gt_b = ((gt == 1) & valid).float()
    for i, t in enumerate(thresholds):
        pred = ((probs > t) & valid).float()
        inter[i] = (pred * gt_b).sum().item()
        union[i] = ((pred + gt_b).clamp(0, 1)).sum().item()
    return {float(t): float(inter[i] / max(union[i].item(), 1.0)) for i, t in enumerate(thresholds)}


@torch.inference_mode()
def collect_test_probs(model, loader, cache_path: Path):
    if cache_path.exists():
        return torch.load(cache_path, map_location='cpu')
    model.eval()
    probs_list = []
    for batch in tqdm(loader, desc=f'collect test probs -> {cache_path.name}'):
        images = batch['images'].to(device)
        intr = batch['intrinsics'].to(device)
        c2c = batch['car2cams'].to(device)
        rover_id = batch['rover_id'].to(device)
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda' and cfg['use_amp'])):
            logits = model(images, intr, c2c, rover_id)
        probs = torch.sigmoid(logits.float()).cpu().to(torch.float16)
        probs_list.append(probs)
    probs = torch.cat(probs_list, dim=0)
    torch.save(probs, cache_path)
    return probs


In [ ]:
ds_val = BEVDatasetV4Clean(val_info, mode='val', img_hw=cfg['img_hw'], rover_vocab=rover_vocab)
ds_test = BEVDatasetV4Clean(test_info, mode='test', img_hw=cfg['img_hw'], rover_vocab=rover_vocab)

loader_val = DataLoader(ds_val, batch_size=1, shuffle=False, num_workers=0, pin_memory=(device.type == 'cuda'))
loader_test = DataLoader(ds_test, batch_size=1, shuffle=False, num_workers=2, pin_memory=(device.type == 'cuda'))

print('val:', len(ds_val), 'test:', len(ds_test))


In [ ]:
thresholds = [round(x, 2) for x in np.arange(0.20, 0.82, 0.02)]
results = []

for candidate_name, ckpt_path, use_ema in [
    ('best', BEST_PT, False),
    ('ema_best', EMA_BEST_PT, True),
]:
    cleanup_cuda()
    model = load_model_from_ckpt(ckpt_path, use_ema_if_present=use_ema)
    val_probs, val_gt = collect_val_probs(model, loader_val, OUT_DIR / f'val_probs_{candidate_name}.pt')
    iou_by_t = threshold_sweep_from_cached_probs(val_probs, val_gt, thresholds)
    best_t, best_iou = max(iou_by_t.items(), key=lambda kv: kv[1])
    sweep_df = pd.DataFrame({'threshold': list(iou_by_t.keys()), 'iou': list(iou_by_t.values())})
    sweep_df.to_csv(OUT_DIR / f'threshold_sweep_{candidate_name}.csv', index=False)
    print(candidate_name, 'best_t =', best_t, 'best_iou =', best_iou)
    display(sweep_df.sort_values('iou', ascending=False).head(10))
    results.append({'candidate': candidate_name, 'best_t': float(best_t), 'best_iou': float(best_iou)})
    del model
    cleanup_cuda()

results_df = pd.DataFrame(results)
results_df['ema_priority'] = (results_df['candidate'] == 'ema_best').astype(int)
results_df = results_df.sort_values(['best_iou', 'ema_priority'], ascending=[False, False]).reset_index(drop=True)
display(results_df)


In [ ]:
selected_name = results_df.iloc[0]['candidate']
selected_threshold = float(results_df.iloc[0]['best_t'])
selected_ckpt = BEST_PT if selected_name == 'best' else EMA_BEST_PT
selected_use_ema = (selected_name == 'ema_best')

cleanup_cuda()
model = load_model_from_ckpt(selected_ckpt, use_ema_if_present=selected_use_ema)
test_probs = collect_test_probs(model, loader_test, OUT_DIR / f'test_probs_{selected_name}.pt')
print('selected candidate:', selected_name)
print('selected threshold:', selected_threshold)
print('test_probs shape:', tuple(test_probs.shape))
del model
cleanup_cuda()


In [ ]:
def pred_name_from_row(row):
    if 'predicted_occupancy_grid' in row:
        return Path(row['predicted_occupancy_grid']).name
    if 'predicted_static_grid' in row:
        return Path(row['predicted_static_grid']).name
    return f'{int(row.name)}.npy'


def make_submission_from_probs(test_probs: torch.Tensor, threshold: float, tag: str):
    thr_tag = f'{threshold:.2f}'.replace('.', 'p')
    pred_dir = OUT_DIR / f'predicted_static_grids_{tag}_{thr_tag}'
    pred_dir.mkdir(parents=True, exist_ok=True)

    for p in pred_dir.glob('*.npy'):
        p.unlink()

    preds = (test_probs > threshold).numpy().astype(np.int32)
    assert len(preds) == len(test_info), (len(preds), len(test_info))

    for i, row in test_info.iterrows():
        np.save(pred_dir / pred_name_from_row(row), preds[i].reshape(1, BEV_H, BEV_W))

    zip_path = OUT_DIR / f'submission_{tag}_t_{thr_tag}.zip'
    if zip_path.exists():
        zip_path.unlink()

    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
        zf.write(DATA_TEST / 'info.csv', arcname='info.csv')
        for npy in sorted(pred_dir.glob('*.npy')):
            zf.write(npy, arcname=f'predicted_static_grids/{npy.name}')

    with zipfile.ZipFile(zip_path, 'r') as zf:
        bad = zf.testzip()
        assert bad is None, bad

    sha = hashlib.sha256(zip_path.read_bytes()).hexdigest()
    result = {
        'threshold': float(threshold),
        'zip_path': str(zip_path.resolve()),
        'size_mb': round(zip_path.stat().st_size / 1e6, 3),
        'sha256': sha,
    }
    print(json.dumps(result, indent=2, ensure_ascii=False))
    return result


In [ ]:
submit_thresholds = sorted(set([
    round(selected_threshold - 0.04, 2),
    round(selected_threshold - 0.02, 2),
    round(selected_threshold, 2),
    round(selected_threshold + 0.02, 2),
    round(selected_threshold + 0.04, 2),
]))
submit_thresholds = [t for t in submit_thresholds if 0.05 <= t <= 0.95]
print('submit_thresholds:', submit_thresholds)

submission_results = []
for t in submit_thresholds:
    print('\n' + '=' * 100)
    print(f'building submission for threshold={t:.2f}')
    submission_results.append(make_submission_from_probs(test_probs, t, selected_name))

submission_results_df = pd.DataFrame(submission_results)
display(submission_results_df)


In [ ]:
bundle_path = OUT_DIR / 'all_submissions_bundle.zip'
if bundle_path.exists():
    bundle_path.unlink()

with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    manifest = []
    for row in submission_results:
        p = Path(row['zip_path'])
        zf.write(p, arcname=p.name)
        manifest.append(row)
    zf.writestr('manifest.json', json.dumps(manifest, indent=2, ensure_ascii=False))

print('bundle:', bundle_path)
print('bundle size_mb:', round(bundle_path.stat().st_size / 1e6, 3))
